# JarvisLM-350M: Colab A100 smoke training

目标：在 **保持 353.5M 模型架构不变** 的前提下，验证 A100 能完成 FineWeb-Edu 数据准备、bf16 前反向传播、Muon + AdamW 更新与 checkpoint 保存。

成功条件：GPU 是 A100；训练完成 10 个 update；loss、grad norm 与 tokens/s 均为有限数；产生 `last.pt`。这不是 10B-token 正式训练，不能据此报告最终 PPL、吞吐或 MFU。

In [1]:
# Colab runtime: Runtime -> Change runtime type -> A100 GPU
import torch

assert torch.cuda.is_available(), '请在 Colab 中选择 GPU runtime。'
gpu_name = torch.cuda.get_device_name(0)
assert 'A100' in gpu_name.upper(), f'此 notebook 预期 A100，当前是: {gpu_name}'
print({'gpu': gpu_name, 'memory_gb': round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'bf16': torch.cuda.is_bf16_supported()})

{'gpu': 'NVIDIA A100-SXM4-40GB', 'memory_gb': 39.5, 'bf16': True}


## 安装项目

此 cell 可重复运行。它 clone GitHub 的 `main`，因此先将当前本地改动 commit 并 push，或将 clone URL 改成包含这些改动的分支。公开 FineWeb-Edu 不需要 HF token；只有打开 W&B 时才需要在 Colab Secrets 或环境变量中设置 `WANDB_API_KEY`（项目也兼容 `WANDB_TOKEN`）。

In [4]:
%cd /content
!if [ -d small-llm-from-scratch/.git ]; then git -C small-llm-from-scratch fetch origin codex/jarvislm-350m-training && git -C small-llm-from-scratch checkout codex/jarvislm-350m-training && git -C small-llm-from-scratch pull --ff-only; else git clone --branch codex/jarvislm-350m-training https://github.com/JarvisZhang24/small-llm-from-scratch.git; fi
%cd /content/small-llm-from-scratch
!python -m pip install -q -e '.[dev]'
!PYTHONPATH=src python -c "import torch, jarvislm; print('torch:', torch.__version__)"

/content
Cloning into 'small-llm-from-scratch'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 82 (delta 15), reused 78 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 57.98 KiB | 19.33 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/small-llm-from-scratch
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 107.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 31.1 MB/s eta 0:0

## 固定本次 smoke 配置

模型仍是 24 层、1024 hidden、1024 context 的 350M 基线。仅为了适配 Colab 时长与显存，测试改为 micro-batch=2、梯度累积=8、10 updates，并先关闭 `torch.compile`。正式 H100 配置仍是 16 × 32。

In [5]:
from pathlib import Path

RUN_ROOT = Path('/content/jarvislm-a100-smoke')
TRAIN_DIR = RUN_ROOT / 'data/train'
VAL_DIR = RUN_ROOT / 'data/val'
CHECKPOINT_DIR = RUN_ROOT / 'checkpoints'
TRAIN_TOKENS = 1_000_000
VAL_TOKENS = 100_000
MAX_STEPS = 10
MICRO_BATCH = 2
GRAD_ACCUM = 8
print({'tokens_per_update': 1024 * MICRO_BATCH * GRAD_ACCUM, 'run_root': str(RUN_ROOT)})

{'tokens_per_update': 16384, 'run_root': '/content/jarvislm-a100-smoke'}


## 准备小规模、互不重叠的 FineWeb-Edu shards

首次运行会下载并 tokenize 110 万 token，通常需要几分钟。若目录已经存在，训练器会拒绝覆盖；需要重来时请换 `RUN_ROOT`，不要在 notebook 中静默删除数据。

In [6]:
!PYTHONPATH=src python -m jarvislm.training.train --prepare-data --prepare-only --data-dir {TRAIN_DIR} --val-dir {VAL_DIR} --prepare-train-tokens {TRAIN_TOKENS} --prepare-val-tokens {VAL_TOKENS}

README.md: 100% 26.4k/26.4k [00:00<00:00, 67.1MB/s]
Resolving data files: 100% 2410/2410 [00:00<00:00, 70623.02it/s]
Resolving data files: 100% 140/140 [00:00<00:00, 3779.11it/s]
FineWeb-Edu shard preparation completed
Fatal Python error: PyGILState_Release: thread state 0x7dd65c001d30 must be current when releasing
Python runtime state: finalizing (tstate=0x0000000000b8a5b0)

Thread 0x00007dd7f182a280 (most recent call first):
  <no Python frame>

Extension modules: numpy._core._multiarray_umath, numpy._core._multiarray_tests, numpy.linalg._umath_linalg, torch._C, torch._C._dynamo.autograd_compiler, torch._C._dynamo.eval_frame, torch._C._dynamo.guards, torch._C._dynamo.utils, torch._C._fft, torch._C._linalg, torch._C._nested, torch._C._nn, torch._C._sparse, torch._C._special, _brotli, zstandard.backend_c, simplejson._speedups, charset_normalizer.md, charset_normalizer.cd, pyarrow.lib, numpy.random._common, numpy.random.bit_generator, numpy.random._bounded_integers, numpy.random._mt199

## A100 350M smoke training

这里显式使用 `--no-require-h100`，它只放宽硬件检查，不会改变 350M 模型结构。先使用 `--no-compile` 降低第一次排错难度；这步成功后再进行 compile 检查。W&B 默认为关闭，避免训练验证被账户配置阻塞。

In [ ]:
!PYTHONPATH=src python -m jarvislm.training.train --data-dir {TRAIN_DIR} --val-dir {VAL_DIR} --checkpoint-dir {CHECKPOINT_DIR} --max-steps {MAX_STEPS} --micro-batch-size {MICRO_BATCH} --grad-accumulation-steps {GRAD_ACCUM} --num-workers 2 --no-require-h100 --no-compile --no-wandb --no-resume

In [ ]:
import torch

checkpoint = torch.load(CHECKPOINT_DIR / 'last.pt', map_location='cpu', weights_only=False)
summary = {
    'completed_step': checkpoint['step'],
    'parameter_tensors': len(checkpoint['model']),
    'has_muon': 'muon' in checkpoint,
    'has_ema': 'ema' in checkpoint,
}
assert summary['completed_step'] == MAX_STEPS
summary

## 可选：验证 `torch.compile`

确认上一阶段成功后，把下面 cell 的 `--no-compile` 删除，并改用新的 checkpoint 目录跑 2 steps。compile 首次编译会显著增加第一步时间，所以不要把它当作吞吐结果。

In [ ]:
# 可选：取消下一行开头的 # 后运行。
# !PYTHONPATH=src python -m jarvislm.training.train --data-dir {TRAIN_DIR} --val-dir {VAL_DIR} --checkpoint-dir {RUN_ROOT / 'compiled-checkpoints'} --max-steps 2 --micro-batch-size {MICRO_BATCH} --grad-accumulation-steps {GRAD_ACCUM} --num-workers 2 --no-require-h100 --compile --no-wandb --no-resume

## 记录结果与下一步

记录：GPU 型号、显存、每一步 loss/grad norm/tokens/s、是否保存 checkpoint。

若成功：下一阶段是在同一 A100 上增加到 100 steps，再迁移到 H100 的正式 10B-token 配置。若 OOM：只将 `MICRO_BATCH` 降到 1，并相应提高 `GRAD_ACCUM`；不要改动模型宽度、层数或 context。